# Rich model representations

A discopt {py:class}`~discopt.modeling.core.Model` renders itself as a typeset optimization problem in standard form {cite:p}`Williams2013` — `minimize f(x)` / `subject to g(x) ≤ 0` / variable domains. In Jupyter the model displays automatically; you can also export the LaTeX or HTML for papers and docs.

Let us build a small mixed-integer nonlinear program and just *look* at it.

In [1]:
import discopt.modeling as dm

m = dm.Model("blend")
x = m.continuous("x", lb=0, ub=10)
y = m.binary("y")
m.minimize((x - 3) ** 2 + 2 * y)
m.subject_to(x + 5 * y >= 4)
m.subject_to(dm.exp(x) <= 100)

m  # renders as a typeset problem (HTML/LaTeX via MathJax)

Model: blend
  Variables: 2 (1 continuous, 1 integer/binary)
  Constraints: 2
  Objective: minimize (((x - 3) ** 2) + (2 * y))
  Parameters: 0

The constraints are shown in their natural form (`x + 5y ≥ 4`), even though they are stored internally in normalised `g(x) ≤ 0` form. To get the markup directly — e.g. to paste into a paper — use `to_latex()`:

In [2]:
print(m.to_latex())

\begin{aligned}
& \text{minimize} \quad && \left(x - 3\right)^{2} + 2 \cdot y \\
& \text{subject to} \quad && 4 \le x + 5 \cdot y \\
&  \quad && \exp\!\left(x\right) \le 100 \\
& \text{with} \quad && 0 \le x \le 10 \\
&  \quad && y \in \{0, 1\} \\
\end{aligned}


## Nonlinear expressions

The renderer handles powers (superscripts), division (fractions), and the elementary functions (`exp`, `log`, `sqrt`, the trig family, …).

In [3]:
n = dm.Model("nlp")
a = n.continuous("a", lb=1, ub=5)
b = n.continuous("b", lb=1, ub=5)
n.maximize(dm.exp(a) / (b + 1))
n.subject_to(dm.sqrt(a) + b**2 <= 10)
n

Model: nlp
  Variables: 2 (2 continuous, 0 integer/binary)
  Constraints: 1
  Objective: maximize (exp(a) / (b + 1))
  Parameters: 0

## Large models are summarised

Auto-display truncates large models so a notebook is never flooded: the first few constraints are shown, then a `⋮` row with the totals, and variables are summarised by type. Pass `max_rows=None` to `to_latex()` / `to_html()` for the complete form.

In [4]:
big = dm.Model("assignment")
z = [big.binary(f"z{i}") for i in range(40)]
big.minimize(sum((i + 1) * z[i] for i in range(40)))
for i in range(40):
    big.subject_to(z[i] + z[(i + 1) % 40] <= 1)
big  # truncated summary

Model: assignment
  Variables: 40 (0 continuous, 40 integer/binary)
  Constraints: 40
  Objective: minimize ((((((((((((((((((((((((((((((((((((((((0 + (1 * z0)) + (2 * z1)) + (3 * z2)) + (4 * z3)) + (5 * z4)) + (6 * z5)) + (7 * z6)) + (8 * z7)) + (9 * z8)) + (10 * z9)) + (11 * z10)) + (12 * z11)) + (13 * z12)) + (14 * z13)) + (15 * z14)) + (16 * z15)) + (17 * z16)) + (18 * z17)) + (19 * z18)) + (20 * z19)) + (21 * z20)) + (22 * z21)) + (23 * z22)) + (24 * z23)) + (25 * z24)) + (26 * z25)) + (27 * z26)) + (28 * z27)) + (29 * z28)) + (30 * z29)) + (31 * z30)) + (32 * z31)) + (33 * z32)) + (34 * z33)) + (35 * z34)) + (36 * z35)) + (37 * z36)) + (38 * z37)) + (39 * z38)) + (40 * z39))
  Parameters: 0

## Exporting

`to_latex()` returns a LaTeX `aligned` block; `to_html()` returns a self-contained HTML fragment (the LaTeX typeset via MathJax) with a header naming the model. Both take `max_rows` to control truncation. The plain-text `repr` / `Model.summary()` remain available for terminals and logging.

In [5]:
print(m.to_html()[:300], "...")

<div class="discopt-model"><div style="font-weight:600">Model <code>blend</code> <span style="color:#888;font-weight:400">(2 variables, 2 constraints)</span></div>$$
\begin{aligned}
& \text{minimize} \quad && \left(x - 3\right)^{2} + 2 \cdot y \\
& \text{subject to} \quad && 4 \le x + 5 \cdot y \\
& ...
